In [1]:
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

print("Imports loaded.")

Imports loaded.


In [2]:
pd.options.display.float_format = "{:,.2f}".format

In [3]:
def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(5):
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found. Run Jupyter from project root or keep /data and /notebooks in place.")

In [4]:
NOTEBOOK_PATH = Path.cwd()

PROJECT_ROOT = find_project_root(NOTEBOOK_PATH)

RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

filtered_file = PROCESSED_DIR / "AIS_2024_01_panynj_scope.csv"

print("Loading scoped dataset...")
print("File exists:", filtered_file.exists())

AIS_panynj = pd.read_csv(filtered_file)

print("\nDataset loaded.")
display(AIS_panynj.shape)
display(AIS_panynj.head())

Loading scoped dataset...
File exists: True

Dataset loaded.


(10856366, 17)

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass
0,366952790,2024-01-01T00:00:00,40.70,-74.01,0.40,303.00,160.00,GUY V MOLINARI,IMO9333486,WDB8295,60.00,0.00,100.00,24.00,4.00,60.00,A
1,368327980,2024-01-01T00:00:01,40.64,-74.11,0.00,360.00,511.00,HENRY GIRLS,NaN,WDN9539,52.00,0.00,75.00,26.00,0.00,52.00,A
2,367390740,2024-01-01T00:00:06,40.95,-73.07,0.00,0.00,511.00,SEAWOLF,IMO8853489,WDE7235,90.00,0.00,24.00,6.00,3.50,90.00,A
3,367409290,2024-01-01T00:00:07,43.21,-70.45,10.60,212.10,212.00,TIMOTHY L REINAUER,IMO7902049,WDE8763,52.00,0.00,138.00,24.00,8.30,57.00,A
4,367022790,2024-01-01T00:00:04,40.53,-74.25,0.00,305.30,199.00,QUENAMES,NaN,WTU9038,52.00,0.00,25.00,9.00,3.00,52.00,A


In [5]:
# Ensure datetime type
AIS_panynj["BaseDateTime"] = pd.to_datetime(AIS_panynj["BaseDateTime"])

print("Time range check:\n")

display(
    AIS_panynj["BaseDateTime"].agg([
        "min",
        "max"
    ])
)

print("\nUnique years:")
display(AIS_panynj["BaseDateTime"].dt.year.unique())

print("\nUnique months:")
display(AIS_panynj["BaseDateTime"].dt.month.unique())

Time range check:



min   2024-01-01 00:00:00
max   2024-01-31 23:59:59
Name: BaseDateTime, dtype: datetime64[us]


Unique years:


array([2024], dtype=int32)


Unique months:


array([1], dtype=int32)

In [6]:
AIS_panynj["Day"] = AIS_panynj["BaseDateTime"].dt.day
AIS_panynj["TimeOfDay"] = AIS_panynj["BaseDateTime"].dt.time

AIS_panynj = AIS_panynj.drop(columns=["BaseDateTime"])

print("BaseDateTime split complete.\n")

display(AIS_panynj[["Day", "TimeOfDay"]].head())

print("\nUpdated shape:")
display(AIS_panynj.shape)

BaseDateTime split complete.



,Day,TimeOfDay
0,1,00:00:00
1,1,00:00:01
2,1,00:00:06
3,1,00:00:07
4,1,00:00:04



Updated shape:


(10856366, 18)

In [7]:
print("Dataset structure:\n")

display(AIS_panynj.info())

print("\nShape:")
display(AIS_panynj.shape)


Dataset structure:

<class 'pandas.DataFrame'>
RangeIndex: 10856366 entries, 0 to 10856365
Data columns (total 18 columns):
 #   Column            Dtype  
---  ------            -----  
 0   MMSI              int64  
 1   LAT               float64
 2   LON               float64
 3   SOG               float64
 4   COG               float64
 5   Heading           float64
 6   VesselName        str    
 7   IMO               str    
 8   CallSign          str    
 9   VesselType        float64
 10  Status            float64
 11  Length            float64
 12  Width             float64
 13  Draft             float64
 14  Cargo             float64
 15  TransceiverClass  str    
 16  Day               int32  
 17  TimeOfDay         object 
dtypes: float64(11), int32(1), int64(1), object(1), str(4)
memory usage: 1.7+ GB


None


Shape:


(10856366, 18)

In [8]:
DTYPE_MAP = {
    # Core identifiers
    "MMSI": "int64",

    # Coordinates
    "LAT": "float64",
    "LON": "float64",

    # Navigation metrics
    "SOG": "float64",
    "COG": "float64",
    "Heading": "float64",

    # Vessel identity (keep as string for now)
    "VesselName": "string",
    "IMO": "string",
    "CallSign": "string",

    # Coded / dimensional attributes
    "VesselType": "Int64",
    "Status": "Int64",
    "Cargo": "Int64",

    # Physical dimensions
    "Length": "Int64",
    "Width": "Int64",
    "Draft": "float64",

    # Device / transmission
    "TransceiverClass": "string",

    # Derived / temporal
    "Day": "int32",
    "TimeOfDay": "string"
}

AIS_panynj = AIS_panynj.astype(DTYPE_MAP)
display(AIS_panynj.info())

<class 'pandas.DataFrame'>
RangeIndex: 10856366 entries, 0 to 10856365
Data columns (total 18 columns):
 #   Column            Dtype  
---  ------            -----  
 0   MMSI              int64  
 1   LAT               float64
 2   LON               float64
 3   SOG               float64
 4   COG               float64
 5   Heading           float64
 6   VesselName        string 
 7   IMO               string 
 8   CallSign          string 
 9   VesselType        Int64  
 10  Status            Int64  
 11  Length            Int64  
 12  Width             Int64  
 13  Draft             float64
 14  Cargo             Int64  
 15  TransceiverClass  string 
 16  Day               int32  
 17  TimeOfDay         string 
dtypes: Int64(5), float64(6), int32(1), int64(1), string(5)
memory usage: 1.8 GB


None

In [9]:
print(f"Trimming whitespace from string columns...\n{'=' * 40}")

string_cols = [
    "VesselName",
    "IMO",
    "CallSign",
    "TransceiverClass",
    "TimeOfDay",
]

for col in string_cols:
    AIS_panynj[col] = (
        AIS_panynj[col]
        .astype("string")
        .str.strip()
    )

print("Whitespace trimming completed.")
print(f"Columns processed: {string_cols}\n{'=' * 40}")

display(AIS_panynj[string_cols].head())

Trimming whitespace from string columns...
Whitespace trimming completed.
Columns processed: ['VesselName', 'IMO', 'CallSign', 'TransceiverClass', 'TimeOfDay']


,VesselName,IMO,CallSign,TransceiverClass,TimeOfDay
0,GUY V MOLINARI,IMO9333486,WDB8295,A,00:00:00
1,HENRY GIRLS,<NA>,WDN9539,A,00:00:01
2,SEAWOLF,IMO8853489,WDE7235,A,00:00:06
3,TIMOTHY L REINAUER,IMO7902049,WDE8763,A,00:00:07
4,QUENAMES,<NA>,WTU9038,A,00:00:04


In [10]:
print(f"Standardising casing for identifier columns...\n{'=' * 40}")

case_cols = [
    "VesselName",
    "IMO",
    "CallSign",
    "TransceiverClass",
]

for col in case_cols:
    AIS_panynj[col] = (
        AIS_panynj[col]
        .astype("string")
        .str.upper()
    )

print("Casing standardisation completed.")
print(f"Columns processed: {case_cols}\n{'=' * 40}")

display(AIS_panynj[case_cols].head())

Standardising casing for identifier columns...
Casing standardisation completed.
Columns processed: ['VesselName', 'IMO', 'CallSign', 'TransceiverClass']


,VesselName,IMO,CallSign,TransceiverClass
0,GUY V MOLINARI,IMO9333486,WDB8295,A
1,HENRY GIRLS,<NA>,WDN9539,A
2,SEAWOLF,IMO8853489,WDE7235,A
3,TIMOTHY L REINAUER,IMO7902049,WDE8763,A
4,QUENAMES,<NA>,WTU9038,A


In [11]:
display(AIS_panynj.sample(20, random_state=42))

,MMSI,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,Day,TimeOfDay
6182096,366853410,40.75,-74.01,23.60,215.80,511.00,FRED V MORRONE,<NA>,WDA8350,60,0,21,6,2.00,60,A,18,21:13:23
5229179,249973000,39.66,-73.88,21.30,165.90,165.00,MSC MERAVIGLIA,IMO9760512,9HA4455,69,0,316,52,8.60,69,A,16,02:01:11
8364176,366982270,37.76,-75.49,5.00,198.70,511.00,CASPIAN DAWN,IMO7932379,WCY6659,52,0,22,8,2.80,52,A,24,22:56:19
475503,367351520,40.70,-74.15,0.00,151.00,316.00,BRIAN NICHOLAS,IMO8851297,WDE4333,52,12,23,8,3.00,52,A,2,11:10:32
2936274,367112860,40.64,-74.07,0.10,360.00,511.00,NICHOLAS MILLER,<NA>,WDD2246,53,0,10,12,0.00,0,A,9,11:19:18
1637140,367147860,40.97,-73.50,6.60,64.70,64.00,CHOPTANK,IMO9417074,WDG4555,31,11,30,11,4.20,31,A,5,19:12:41
719886,636093060,29.18,-94.62,0.00,263.00,86.00,BRUCE,IMO9741140,D5LJ6,70,1,147,23,7.60,70,A,3,03:31:49
212552,367006540,40.66,-74.04,0.10,249.60,215.00,THE BEATRICE,IMO8117964,WDM6451,52,1,157,24,5.50,57,A,1,17:43:43
5484649,311040800,30.40,-81.35,12.20,100.10,100.00,HARMONY LEADER,IMO9441568,C6YP3,70,0,200,32,9.50,70,A,16,19:48:33
9466071,367645150,40.64,-74.15,0.00,179.20,511.00,STEPHEN B,IMO8334081,WDH7135,31,12,86,7,3.00,31,A,27,19:18:56


In [12]:
# --- Sentinel value handling (from profiling rules) ---


AIS_panynj["SOG"] = AIS_panynj["SOG"].replace(102.3, np.nan)
AIS_panynj["COG"] = AIS_panynj["COG"].replace(360, np.nan)
AIS_panynj["Heading"] = AIS_panynj["Heading"].replace(511, np.nan)

AIS_panynj["Length"] = AIS_panynj["Length"].replace(0, np.nan)
AIS_panynj["Width"] = AIS_panynj["Width"].replace(0, np.nan)
AIS_panynj["Draft"] = AIS_panynj["Draft"].replace(0, np.nan)

AIS_panynj["Cargo"] = AIS_panynj["Cargo"].replace(0, np.nan)

Duplicate Handling

Duplicate transmissions were identified using the composite key:

MMSI + BaseDateTime

Duplicates were verified to be exact record repetitions and were removed by retaining the first occurrence of each duplicate group.

This ensures that each vessel has at most one transmission per timestamp, preserving temporal integrity for movement and interval calculations.

In [13]:
dup_mask = AIS_panynj.duplicated(
    subset=["MMSI", "Day", "TimeOfDay"],
    keep=False
)

duplicates = AIS_panynj[dup_mask]

print("Total duplicate rows:", len(duplicates))
print("Unique duplicate timestamps:", duplicates[["MMSI", "Day", "TimeOfDay"]].drop_duplicates().shape[0])

Total duplicate rows: 950
Unique duplicate timestamps: 475


In [14]:
total_rows = len(AIS_panynj)
duplicate_rows = dup_mask.sum()

print("Duplicate rows:", duplicate_rows)
print("Duplicate rate:", duplicate_rows / total_rows)

Duplicate rows: 950
Duplicate rate: 8.750626130327589e-05


In [15]:
duplicates.head(20)

,MMSI,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,Day,TimeOfDay
53275,368029640,40.75,-74.02,4.90,187.60,NaN,THE MANHATTAN II,<NA>,WDJ9810,60,5,36,10,NaN,60,A,1,03:59:59
64128,368029640,40.75,-74.02,4.90,187.60,NaN,THE MANHATTAN II,<NA>,WDJ9810,60,5,36,10,NaN,60,A,1,03:59:59
78883,368056750,40.65,-74.10,3.20,63.20,53.00,AVA M MCALLISTER,IMO9857561,WDK4592,52,0,33,12,NaN,52,A,1,06:00:00
133697,338725000,40.66,-74.03,0.00,286.40,13.00,CAPE HATTERAS,IMO9877597,WDK7639,57,1,126,24,5.10,57,A,1,11:00:00
144691,338458000,40.64,-72.96,10.20,250.10,249.00,CAPE LOOKOUT,IMO9857353,WDK2536,57,0,147,24,4.90,57,A,1,11:59:59
145134,338458000,40.64,-72.96,10.20,250.10,249.00,CAPE LOOKOUT,IMO9857353,WDK2536,57,0,147,24,4.90,57,A,1,11:59:59
168533,367091270,40.44,-74.02,10.30,170.90,NaN,CAPTAIN TOM,IMO0000000,WBX3591,33,<NA>,14,6,NaN,<NA>,B,1,14:00:00
179008,563032900,32.00,-80.27,13.10,250.80,250.00,MAERSK SEOUL,IMO9306550,9V5632,71,0,332,43,12.10,71,A,1,15:00:00
179014,563032900,32.00,-80.27,13.10,250.80,250.00,MAERSK SEOUL,IMO9306550,9V5632,71,0,332,43,12.10,71,A,1,15:00:00
179836,367431620,41.29,-72.91,0.00,325.80,10.00,MOUNT ST ELIAS,IMO8749884,WDI8452,31,5,125,24,6.10,57,A,1,13:00:00


In [16]:
duplicates.groupby(
    ["MMSI", "Day", "TimeOfDay"]
).nunique().sort_values(
    by="LAT",
    ascending=False
).head()

,,,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass
MMSI,Day,TimeOfDay,,,,,,,,,,,,,,,
366939790,16,21:59:59,2,1,2,2,2,1,1,1,1,1,1,1,1,1,1
367608840,23,06:00:00,2,2,1,1,1,1,1,1,1,1,1,1,1,1,1
563161200,31,11:00:00,2,2,1,1,1,1,1,1,1,1,1,1,1,1,1
215196000,24,10:00:00,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
215801000,26,02:59:59,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [17]:
variation = (
    duplicates.groupby(
        ["MMSI", "Day", "TimeOfDay"]
    ).nunique()
)

conflicting = variation[
    (variation > 1).any(axis=1)
]

exact = variation[
    (variation == 1).all(axis=1)
]

print("Exact duplicate groups:", len(exact))
print("Conflicting duplicate groups:", len(conflicting))

Exact duplicate groups: 289
Conflicting duplicate groups: 5


In [18]:
AIS_cleaned = AIS_panynj.drop_duplicates(
    subset=[
        "MMSI",
        "Day",
        "TimeOfDay",
        "LAT",
        "LON",
        "SOG",
        "COG",
        "Heading",
        "VesselName",
        "IMO",
        "CallSign",
        "VesselType",
        "Status",
        "Length",
        "Width",
        "Draft",
        "Cargo",
        "TransceiverClass"
    ],
    keep="first"
)

print("Original rows:", len(AIS_panynj))
print("Rows after removing exact duplicates:", len(AIS_cleaned))

Original rows: 10856366
Rows after removing exact duplicates: 10855896


In [19]:
# Get the index keys for conflicting groups
conflicting_keys = conflicting.reset_index()[
    ["MMSI", "Day", "TimeOfDay"]
]

# Retrieve the actual duplicate records
conflicting_records = duplicates.merge(
    conflicting_keys,
    on=["MMSI", "Day", "TimeOfDay"],
    how="inner"
)

# Display them clearly
conflicting_records.sort_values(
    ["MMSI", "Day", "TimeOfDay"]
)

,MMSI,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,Day,TimeOfDay
0,366939790,40.54,-74.25,1.20,192.40,265.00,DORIS MORAN,IMO8121800,WRU2125,52,0,36,10,4.60,52,A,16,21:59:59
1,366939790,40.54,-74.25,1.10,192.00,266.00,DORIS MORAN,IMO8121800,WRU2125,52,0,36,10,4.60,52,A,16,21:59:59
2,367147860,42.31,-70.78,9.30,103.30,99.00,CHOPTANK,IMO9417074,WDG4555,31,11,30,11,4.20,31,A,21,14:00:00
3,367147860,42.31,-70.78,9.30,102.90,99.00,CHOPTANK,IMO9417074,WDG4555,31,11,30,11,4.20,31,A,21,14:00:00
4,367608840,40.56,-73.80,5.90,76.90,77.00,CHARLES A,IMO8424202,WDH3586,52,0,26,8,3.30,52,A,23,06:00:00
5,367608840,40.56,-73.80,5.90,76.90,77.00,CHARLES A,IMO8424202,WDH3586,52,0,26,8,3.30,52,A,23,06:00:00
6,367797260,40.65,-74.03,0.20,54.60,124.00,SPRING MALLARD,<NA>,WDJ6389,69,0,26,8,2.00,69,A,26,12:00:00
7,367797260,40.65,-74.03,0.10,45.00,123.00,SPRING MALLARD,<NA>,WDJ6389,69,0,26,8,2.00,69,A,26,12:00:00
8,563161200,29.01,-116.44,13.90,343.30,341.00,EVER FORWARD,IMO9850551,9V7977,74,0,334,48,11.30,74,A,31,11:00:00
9,563161200,29.02,-116.44,13.90,343.30,341.00,EVER FORWARD,IMO9850551,9V7977,74,0,334,48,11.30,74,A,31,11:00:00


In [20]:
# Resolve conflicting duplicates by keeping one deterministic record per group

conflicting_keys = conflicting_records[
    ["MMSI", "Day", "TimeOfDay"]
].drop_duplicates()

conflicting_resolved = (
    conflicting_records
    .sort_values(
        ["MMSI", "Day", "TimeOfDay"]
    )
    .drop_duplicates(
        subset=["MMSI", "Day", "TimeOfDay"],
        keep="first"
    )
)

print("Resolved conflicting groups:", len(conflicting_resolved))
display(conflicting_resolved)

Resolved conflicting groups: 5


,MMSI,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,Day,TimeOfDay
0,366939790,40.54,-74.25,1.20,192.40,265.00,DORIS MORAN,IMO8121800,WRU2125,52,0,36,10,4.60,52,A,16,21:59:59
2,367147860,42.31,-70.78,9.30,103.30,99.00,CHOPTANK,IMO9417074,WDG4555,31,11,30,11,4.20,31,A,21,14:00:00
4,367608840,40.56,-73.80,5.90,76.90,77.00,CHARLES A,IMO8424202,WDH3586,52,0,26,8,3.30,52,A,23,06:00:00
6,367797260,40.65,-74.03,0.20,54.60,124.00,SPRING MALLARD,<NA>,WDJ6389,69,0,26,8,2.00,69,A,26,12:00:00
8,563161200,29.01,-116.44,13.90,343.30,341.00,EVER FORWARD,IMO9850551,9V7977,74,0,334,48,11.30,74,A,31,11:00:00


In [21]:
# Remove original conflicting rows and replace them with resolved ones

AIS_cleaned = AIS_cleaned.merge(
    conflicting_keys,
    on=["MMSI", "Day", "TimeOfDay"],
    how="left",
    indicator=True
)

# Keep only rows that are not part of conflicting groups
AIS_cleaned = AIS_cleaned.loc[
    AIS_cleaned["_merge"] == "left_only"
].drop(columns="_merge")

# Add back the resolved rows
AIS_cleaned = pd.concat(
    [AIS_cleaned, conflicting_resolved],
    ignore_index=True
)

# Final ordering
AIS_cleaned = AIS_cleaned.sort_values(
    ["MMSI", "Day", "TimeOfDay"]
).reset_index(drop=True)

print("Final cleaned dataset shape:", AIS_cleaned.shape)

Final cleaned dataset shape: (10855891, 18)


In [22]:
mmsi_length = AIS_cleaned["MMSI"].astype("string").str.len()

invalid_mmsi_mask = mmsi_length != 9

print("Malformed MMSI rows:", invalid_mmsi_mask.sum())

invalid_mmsi = AIS_cleaned.loc[invalid_mmsi_mask].copy()

display(invalid_mmsi.head(20))

Malformed MMSI rows: 0


,MMSI,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,Day,TimeOfDay


In [23]:
mmsi_length = AIS_cleaned["MMSI"].astype("string").str.len()

display(
    mmsi_length.value_counts().sort_index()
)

MMSI
9    10855891
Name: count, dtype: Int64

In [24]:
metadata_cols = [
    "VesselName",
    "CallSign",
    "IMO",
    "VesselType",
    "Length",
    "Width",
    "Draft",
    "TransceiverClass"
]

vessel_metadata = (
    AIS_cleaned.groupby("MMSI")[metadata_cols]
    .agg(lambda s: s.dropna().iloc[0] if not s.dropna().empty else pd.NA)
    .reset_index()
)

print("Unique vessels:", vessel_metadata["MMSI"].nunique())
display(vessel_metadata.head(20))

Unique vessels: 896


,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
0,209697000,CONTSHIP LEO,5BHD6,IMO9403451,74,148,25,8.30,A
1,209850000,ANDROMEDA J,5BKR3,IMO9355422,70,140,22,5.60,A
2,209862000,WARNOW WHALE,5BXD3,IMO9395032,70,166,25,6.90,A
3,210749000,PACIFIC TRADER,5BPJ3,IMO9406922,74,147,23,7.70,A
4,210905000,ARSOS,5BAQ2,IMO9395123,70,166,25,7.40,A
5,211549000,BASLE EXPRESS,DFGN2,IMO9501344,71,366,48,12.80,A
6,211578000,ULSAN EXPRESS,DDOQ2,IMO9613020,71,366,48,13.90,A
7,211839000,CHICAGO EXPRESS,DCUJ2,IMO9295268,74,336,43,11.40,A
8,212276000,YM SEATTLE,C4XC2,IMO9360910,74,260,32,10.00,A
9,212659000,NORDAMELIA,5BPQ4,IMO9724958,71,195,32,9.10,A


In [25]:
metadata_missing_summary = (
    vessel_metadata[metadata_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values in vessel metadata:\n")
display(metadata_missing_summary)

Missing values in vessel metadata:



IMO                 275
Draft               254
CallSign             68
Width                45
Length               37
VesselName           28
VesselType           27
TransceiverClass      0
dtype: int64

In [26]:
core_metadata_cols = [ "VesselName", "VesselType"]

missing_core_mask = vessel_metadata[core_metadata_cols].isna().any(axis=1)

vessels_missing_core = vessel_metadata.loc[missing_core_mask].copy()

print("Vessels missing core metadata:", len(vessels_missing_core))
display(vessels_missing_core.head(20))

Vessels missing core metadata: 28


,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
68,284127690,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,A
168,366081084,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,A
169,366139092,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,A
183,366772794,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,A
199,366897849,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,A
219,366952789,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,A
284,367125654,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,A
307,367191222,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,A
311,367217712,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,A
363,367419072,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,A


In [27]:
missing_core_mmsi = vessels_missing_core["MMSI"]

transmissions_missing_core = AIS_cleaned[
    AIS_cleaned["MMSI"].isin(missing_core_mmsi)
]

print("Transmissions from vessels missing core metadata:",
      len(transmissions_missing_core))

Transmissions from vessels missing core metadata: 1231


In [28]:
rows_before = len(AIS_cleaned)

AIS_cleaned = AIS_cleaned[
    ~AIS_cleaned["MMSI"].isin(missing_core_mmsi)
].copy()

rows_after = len(AIS_cleaned)

print("Rows removed:", rows_before - rows_after)
print("Final dataset size:", rows_after)

Rows removed: 1231
Final dataset size: 10854660


In [29]:
vessel_metadata = (
    AIS_cleaned.groupby("MMSI")[metadata_cols]
    .agg(lambda s: s.dropna().iloc[0] if not s.dropna().empty else pd.NA)
    .reset_index()
)

print("Unique vessels:", vessel_metadata["MMSI"].nunique())

Unique vessels: 868


In [30]:
metadata_missing_summary = (
    vessel_metadata[metadata_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values in vessel metadata:\n")
display(metadata_missing_summary)

Missing values in vessel metadata:



IMO                 247
Draft               227
CallSign             41
Width                18
Length               10
VesselName            0
VesselType            0
TransceiverClass      0
dtype: int64

In [31]:
metadata_missing_pct = (
    vessel_metadata[metadata_cols]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print("Missing percentage in vessel metadata:\n")
display(metadata_missing_pct)

Missing percentage in vessel metadata:



IMO                28.46
Draft              26.15
CallSign            4.72
Width               2.07
Length              1.15
VesselName          0.00
VesselType          0.00
TransceiverClass    0.00
dtype: float64

In [32]:
vessel_dimensions = vessel_metadata.copy()

vessel_dimensions["length_width_ratio"] = (
    vessel_dimensions["Length"] / vessel_dimensions["Width"]
)

vessel_dimensions["draft_length_ratio"] = (
    vessel_dimensions["Draft"] / vessel_dimensions["Length"]
)

vessel_dimensions["draft_width_ratio"] = (
    vessel_dimensions["Draft"] / vessel_dimensions["Width"]
)

issue_masks = {
    "Width > Length": (
        vessel_dimensions["Length"].notna() &
        vessel_dimensions["Width"].notna() &
        (vessel_dimensions["Width"] > vessel_dimensions["Length"])
    ),

    "Length too small (< 5)": (
        vessel_dimensions["Length"].notna() &
        (vessel_dimensions["Length"] < 5)
    ),

    "Width too small (< 2)": (
        vessel_dimensions["Width"].notna() &
        (vessel_dimensions["Width"] < 2)
    ),

    "Length too large (> 400)": (
        vessel_dimensions["Length"].notna() &
        (vessel_dimensions["Length"] > 400)
    ),

    "Width too large (> 80)": (
        vessel_dimensions["Width"].notna() &
        (vessel_dimensions["Width"] > 80)
    ),

    "Draft negative": (
        vessel_dimensions["Draft"].notna() &
        (vessel_dimensions["Draft"] < 0)
    ),

    "Draft too large (> 25)": (
        vessel_dimensions["Draft"].notna() &
        (vessel_dimensions["Draft"] > 25)
    ),

    "Length/Width ratio < 1": (
        vessel_dimensions["Length"].notna() &
        vessel_dimensions["Width"].notna() &
        (vessel_dimensions["length_width_ratio"] < 1)
    ),

    "Length/Width ratio > 15": (
        vessel_dimensions["Length"].notna() &
        vessel_dimensions["Width"].notna() &
        (vessel_dimensions["length_width_ratio"] > 15)
    ),

    "Draft/Length ratio > 0.6": (
        vessel_dimensions["Draft"].notna() &
        vessel_dimensions["Length"].notna() &
        (vessel_dimensions["draft_length_ratio"] > 0.6)
    ),

    "Draft/Width ratio > 2": (
        vessel_dimensions["Draft"].notna() &
        vessel_dimensions["Width"].notna() &
        (vessel_dimensions["draft_width_ratio"] > 2)
    ),

    "Only one dimension present": (
        (vessel_dimensions["Length"].notna() & vessel_dimensions["Width"].isna()) |
        (vessel_dimensions["Length"].isna() & vessel_dimensions["Width"].notna())
    )
}

In [33]:
issue_summary_rows = []

for issue_name, issue_mask in issue_masks.items():
    flagged_vessels = vessel_dimensions.loc[issue_mask, "MMSI"]
    flagged_transmissions = AIS_cleaned["MMSI"].isin(flagged_vessels).sum()

    issue_summary_rows.append({
        "Issue": issue_name,
        "AffectedVessels": flagged_vessels.nunique(),
        "AffectedTransmissions": flagged_transmissions,
        "MMSI_List": flagged_vessels.drop_duplicates().tolist()
    })

issue_summary = (
    pd.DataFrame(issue_summary_rows)
    .sort_values(
        ["AffectedVessels", "AffectedTransmissions"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(issue_summary)

,Issue,AffectedVessels,AffectedTransmissions,MMSI_List
0,Only one dimension present,8,16324,"[338120906, 367407270, 367627870, 367713330, 3..."
1,Width > Length,2,35515,"[367112860, 367541190]"
2,Length/Width ratio < 1,2,35515,"[367112860, 367541190]"
3,Length too small (< 5),2,5435,"[367627870, 367713330]"
4,Draft too large (> 25),1,384,[368016020]
5,Draft/Width ratio > 2,1,384,[368016020]
6,Width too small (< 2),0,0,[]
7,Length too large (> 400),0,0,[]
8,Width too large (> 80),0,0,[]
9,Draft negative,0,0,[]


In [34]:
active_issue_masks = {
    issue_name: issue_mask
    for issue_name, issue_mask in issue_masks.items()
    if issue_mask.sum() > 0
}

issue_flag_cols = []

for issue_name, issue_mask in active_issue_masks.items():
    col_name = f"Issue_{issue_name}"
    vessel_dimensions[col_name] = issue_mask
    issue_flag_cols.append(col_name)

any_issue_mask = vessel_dimensions[issue_flag_cols].any(axis=1)
dimension_review_cols = [
    "MMSI",
    "VesselName",
    "VesselType",
    "Length",
    "Width",
    "Draft",
    "TransceiverClass",
    "length_width_ratio",
    "draft_length_ratio",
    "draft_width_ratio"
]
suspicious_dimensions_all = vessel_dimensions.loc[
    any_issue_mask,
    dimension_review_cols + issue_flag_cols
].copy()

print("Active dimension issues:", list(active_issue_masks.keys()))
print("Total vessels flagged by at least one active dimension rule:", len(suspicious_dimensions_all))

display(suspicious_dimensions_all)

Active dimension issues: ['Width > Length', 'Length too small (< 5)', 'Draft too large (> 25)', 'Length/Width ratio < 1', 'Draft/Width ratio > 2', 'Only one dimension present']
Total vessels flagged by at least one active dimension rule: 11


,MMSI,VesselName,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio,draft_length_ratio,draft_width_ratio,Issue_Width > Length,Issue_Length too small (< 5),Issue_Draft too large (> 25),Issue_Length/Width ratio < 1,Issue_Draft/Width ratio > 2,Issue_Only one dimension present
95,338120906,FREEBYRD,37,12,<NA>,<NA>,B,<NA>,<NA>,<NA>,False,False,False,False,False,True
274,367112860,NICHOLAS MILLER,53,10,12,<NA>,A,0.83,<NA>,<NA>,True,False,False,True,False,False
349,367407270,WASHINGTON,31,25,<NA>,<NA>,A,<NA>,<NA>,<NA>,False,False,False,False,False,True
399,367541190,RV HENRY HUDSON,34,14,16,<NA>,B,0.88,<NA>,<NA>,True,False,False,True,False,False
432,367627870,SJSO SENTINEL,55,2,<NA>,1.00,A,<NA>,0.50,<NA>,False,True,False,False,False,True
466,367713330,TRISHA MARIE,30,4,<NA>,<NA>,B,<NA>,<NA>,<NA>,False,True,False,False,False,True
515,368004280,COSMO,69,18,<NA>,<NA>,A,<NA>,<NA>,<NA>,False,False,False,False,False,True
523,368016020,TIMELESS,71,50,10,25.50,A,5.00,0.51,2.55,False,False,True,False,True,False
555,368111610,FDNY M8A,51,11,<NA>,<NA>,A,<NA>,<NA>,<NA>,False,False,False,False,False,True
621,368284080,ENCHANTED,60,25,<NA>,<NA>,B,<NA>,<NA>,<NA>,False,False,False,False,False,True


In [35]:
draft_over_25_mask = issue_masks["Draft too large (> 25)"]

draft_over_25_vessels = vessel_dimensions.loc[
    draft_over_25_mask,
    dimension_review_cols
].copy()

display(draft_over_25_vessels)

,MMSI,VesselName,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio,draft_length_ratio,draft_width_ratio
523,368016020,TIMELESS,71,50,10,25.50,A,5.00,0.51,2.55


In [36]:
small_length_mask = issue_masks["Length too small (< 5)"]

small_length_vessels = vessel_dimensions.loc[
    small_length_mask,
    dimension_review_cols
].copy()

display(small_length_vessels)

,MMSI,VesselName,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio,draft_length_ratio,draft_width_ratio
432,367627870,SJSO SENTINEL,55,2,<NA>,1.00,A,<NA>,0.50,<NA>
466,367713330,TRISHA MARIE,30,4,<NA>,<NA>,B,<NA>,<NA>,<NA>


Small dimensions (Length < 5m) are physically plausible — small workboats, pilot boats, and tenders operate in port areas. Only ~14 out of ~800 vessels affected.

No structural contradictions found. Values retained as reported. Adding a reliability flag would add schema complexity for no real benefit at this stage.

In [37]:
width_length_conflict_mmsi = [367112860, 367541190] 
print("Stored MMSI list for Width/Length structural conflict:") 

width_length_conflict_vessels = vessel_dimensions.loc[
    vessel_dimensions["MMSI"].isin(width_length_conflict_mmsi),
    dimension_review_cols
].copy()

display(width_length_conflict_vessels)

Stored MMSI list for Width/Length structural conflict:


,MMSI,VesselName,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio,draft_length_ratio,draft_width_ratio
274,367112860,NICHOLAS MILLER,53,10,12,<NA>,A,0.83,<NA>,<NA>
399,367541190,RV HENRY HUDSON,34,14,16,<NA>,B,0.88,<NA>,<NA>


In [38]:
for mmsi in width_length_conflict_mmsi:

    original_length = vessel_dimensions.loc[
        vessel_dimensions["MMSI"] == mmsi,
        "Length"
    ].values[0]

    original_width = vessel_dimensions.loc[
        vessel_dimensions["MMSI"] == mmsi,
        "Width"
    ].values[0]

    vessel_dimensions.loc[
        vessel_dimensions["MMSI"] == mmsi,
        "Length"
    ] = original_width

    vessel_dimensions.loc[
        vessel_dimensions["MMSI"] == mmsi,
        "Width"
    ] = original_length

Width > Length is structurally impossible for a vessel. The two affected vessels have values close in magnitude — a likely data entry swap rather than a real measurement error.

Decision: swap Length and Width for these two vessels. This produces consistent proportions.

In [39]:
vessel_dimensions["length_width_ratio"] = (
    vessel_dimensions["Length"] /
    vessel_dimensions["Width"]
)

In [40]:
one_dimension_mask = issue_masks["Only one dimension present"]

one_dimension_vessels = vessel_dimensions.loc[
    one_dimension_mask,
    dimension_review_cols
].copy()

display(one_dimension_vessels)

,MMSI,VesselName,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio,draft_length_ratio,draft_width_ratio
95,338120906,FREEBYRD,37,12,<NA>,<NA>,B,<NA>,<NA>,<NA>
349,367407270,WASHINGTON,31,25,<NA>,<NA>,A,<NA>,<NA>,<NA>
432,367627870,SJSO SENTINEL,55,2,<NA>,1.00,A,<NA>,0.50,<NA>
466,367713330,TRISHA MARIE,30,4,<NA>,<NA>,B,<NA>,<NA>,<NA>
515,368004280,COSMO,69,18,<NA>,<NA>,A,<NA>,<NA>,<NA>
555,368111610,FDNY M8A,51,11,<NA>,<NA>,A,<NA>,<NA>,<NA>
621,368284080,ENCHANTED,60,25,<NA>,<NA>,B,<NA>,<NA>,<NA>
867,856583816,CT PILOT_US,0,42,<NA>,<NA>,A,<NA>,<NA>,<NA>


Dimension review complete.

- Length/Width corrected for 2 vessels (likely data entry swap).
- Remaining null values left as-is — gaps are too small to justify imputation and there is no reliable external data to fill from.

Metadata is ready for downstream use.

In [41]:
print("Coordinate validity check:\n")

invalid_lat_mask = ~AIS_cleaned["LAT"].between(-90, 90)
invalid_lon_mask = ~AIS_cleaned["LON"].between(-180, 180)

print("Invalid LAT rows:", invalid_lat_mask.sum())
print("Invalid LON rows:", invalid_lon_mask.sum())

invalid_coordinates = AIS_cleaned.loc[
    invalid_lat_mask | invalid_lon_mask,
    ["MMSI", "Day", "TimeOfDay", "LAT", "LON"]
].copy()

if len(invalid_coordinates) > 0:
    print("\nRows with invalid coordinates:")
    display(invalid_coordinates.head(20))
else:
    print("\nCoordinate validation passed. No invalid LAT/LON values found.")

Coordinate validity check:

Invalid LAT rows: 0
Invalid LON rows: 0

Coordinate validation passed. No invalid LAT/LON values found.


In [42]:
print("Final integrity checkpoint:\n")

rows_scoped_input = len(AIS_panynj)
rows_final_output = len(AIS_cleaned)
rows_removed_total = rows_scoped_input - rows_final_output

print("Rows in scoped input:", rows_scoped_input)
print("Rows in final cleaned output:", rows_final_output)
print("Total rows removed during cleaning:", rows_removed_total)

core_required_cols = ["MMSI", "Day", "TimeOfDay", "LAT", "LON"]

core_null_summary = (
    AIS_cleaned[core_required_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nNull values in core required fields:")
display(core_null_summary)

schema_summary = pd.DataFrame({
    "Column": AIS_cleaned.columns,
    "Dtype": AIS_cleaned.dtypes.astype("string").values
})

print("\nFinal schema summary:")
display(schema_summary)

Final integrity checkpoint:

Rows in scoped input: 10856366
Rows in final cleaned output: 10854660
Total rows removed during cleaning: 1706

Null values in core required fields:


MMSI         0
Day          0
TimeOfDay    0
LAT          0
LON          0
dtype: int64


Final schema summary:


,Column,Dtype
0,MMSI,int64
1,LAT,float64
2,LON,float64
3,SOG,float64
4,COG,float64
5,Heading,float64
6,VesselName,string
7,IMO,string
8,CallSign,string
9,VesselType,Int64


In [43]:
output_path = PROCESSED_DIR / "AIS_2024_01_panynj_cleaned.csv"

AIS_cleaned.to_csv(
    output_path,
    index=False
)

print("Cleaned dataset exported to:")
print(output_path)
print("Rows exported:", len(AIS_cleaned))

Cleaned dataset exported to:
C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Maritime Logistics Intelligence\Data\Processed\AIS_2024_01_panynj_cleaned.csv
Rows exported: 10854660
